# VISUALIZATION

# Cell 1: setup

In [1]:
import sys
sys.path.append("..")

from pathlib import Path
import pandas as pd
import plotly.express as px

from sql.queries import QUERIES
from utils import db_connection as db

OUT = Path("../documentation")
OUT.mkdir(exist_ok=True)


def q(key):
    return db.run_query(QUERIES[key]["sql"])


def save(fig, name):
    """Write an interactive HTML copy (for README screenshots) and return the figure."""
    fig.write_html(OUT / f"{name}.html", include_plotlyjs="cdn")
    return fig

# Cell 2: wins per team

In [2]:
df = q("q05")
fig = px.bar(df, x="team_name", y="total_wins", text="total_wins", color="team_name",
             title="Total wins per team").update_layout(showlegend=False, xaxis_title=None)
save(fig, "wins_per_team").show()

# Cell 3: toss impact

In [3]:
df = q("q17")
fig = (px.bar(df, x="decision", y="win_pct", text="win_pct", color="decision",
              title="How often does the toss winner win the match? (%)")
       .add_hline(y=50, line_dash="dash")
       .update_layout(showlegend=False))
save(fig, "toss_impact").show()

# Cell 4: home vs away

In [4]:
df = q("q12")
df["home_win_pct"] = (100 * df["home_wins"] / df["home_played"]).round(1)
df["away_win_pct"] = (100 * df["away_wins"] / df["away_played"]).round(1)
long = df.melt(id_vars="team_name", value_vars=["home_win_pct", "away_win_pct"],
               var_name="where", value_name="win_pct")
fig = px.bar(long, x="team_name", y="win_pct", color="where", barmode="group",
             title="Win % at home vs away")
save(fig, "home_vs_away").show()

# Cell 5: runs by batting position

In [5]:
runs = db.run_query("""
    SELECT b.batting_position AS position, b.runs, m.match_format AS format
    FROM batting_performances b JOIN matches m ON m.match_id = b.match_id
    WHERE m.match_format IN ('Test','ODI','T20I')""")
fig = px.box(runs, x="position", y="runs", color="format",
             title="Runs scored by batting position")
save(fig, "runs_by_position").show()

# Cell 6: partnerships

In [6]:
pr = q("q13")
fig = px.histogram(pr, x="partnership_runs", nbins=30,
                   title="Distribution of 100+ partnerships")
save(fig, "partnerships").show()

# Cell 7: one player’s yearly trend

In [7]:
yearly = q("q16")
star = yearly.groupby("full_name")["matches_played"].sum().idxmax()
one = yearly[yearly["full_name"] == star]
fig = px.line(one, x="year", y="avg_runs_per_match", markers=True,
              title=f"Yearly batting trend: {star}")
save(fig, "player_trend").show()

# Cell 8: form categories

In [8]:
form = q("q23")["form"].value_counts().reset_index()
form.columns = ["form", "players"]
fig = px.pie(form, names="form", values="players", hole=0.4,
             title="Player form categories (last 10 innings)")
save(fig, "form_categories").show()

# Cell 9: career phase

In [9]:
phase = q("q25")["career_phase"].value_counts().reset_index()
phase.columns = ["career_phase", "players"]
fig = px.bar(phase, x="career_phase", y="players", text="players",
             title="Career phase distribution")
save(fig, "career_phase").show()

# Cell 10: players by country and role (heatmap)

In [10]:
pc = db.run_query("SELECT country, playing_role, COUNT(*) AS n FROM players "
                  "WHERE country IS NOT NULL AND playing_role IS NOT NULL GROUP BY 1, 2")
pivot = pc.pivot(index="country", columns="playing_role", values="n").fillna(0)
fig = px.imshow(pivot, text_auto=True, aspect="auto", title="Players by country and role")
save(fig, "country_role_heatmap").show()

In [11]:
import sys, json
sys.path.append("..")
from api.cricbuzz_client import CricbuzzClient
import config

data = CricbuzzClient().get(config.ENDPOINTS["live_matches"])   # uses saved copy if present
first = data["typeMatches"][0]["seriesMatches"][0]["seriesAdWrapper"]["matches"][0]
print(json.dumps(first, indent=2)[:2500])

{
  "matchInfo": {
    "matchId": 151602,
    "seriesId": 11902,
    "seriesName": "West Indies tour of India, 2026",
    "matchDesc": "5th T20I",
    "matchFormat": "T20",
    "startDate": "1792243800000",
    "endDate": "1792256400000",
    "state": "In Progress",
    "status": "9 Overs game due to rain",
    "team1": {
      "teamId": 2,
      "teamName": "India",
      "teamSName": "IND",
      "imageId": 776162
    },
    "team2": {
      "teamId": 10,
      "teamName": "West Indies",
      "teamSName": "WI",
      "imageId": 776191
    },
    "venueInfo": {
      "id": 27,
      "ground": "M.Chinnaswamy Stadium",
      "city": "Bengaluru",
      "timezone": "+05:30",
      "latitude": "12.978853",
      "longitude": "77.599533"
    },
    "currBatTeamId": 2,
    "seriesStartDt": "1790467200000",
    "seriesEndDt": "1792368000000",
    "isTimeAnnounced": true,
    "stateTitle": "In Progress",
    "isFantasyEnabled": true,
    "hideScoreStatus": "Match in progress"
  },
  "matchSco